I try to implement temperature in a better way

In [2]:
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
import numpy as np
import threading
from PIL import Image

import matplotlib
matplotlib.use("TkAgg")
from matplotlib import colormaps as mpl_cm
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from scipy.fft import fft2, ifft2, fftfreq


class GPE2DSimulatorApp:
    def __init__(self, root):
        self.root = root
        self.root.title("2D GPE Simulator: Thermal SGPE & Live Recording")
        self.root.geometry("1250x800")
        try:
            self.root.state('zoomed')
        except tk.TclError:
            try:
                self.root.attributes('-zoomed', True)
            except tk.TclError:
                pass

        # ── Spatial grid (x == y by symmetry, so we only build one) ──────────
        self.N = 128
        self.L = 30.0
        x = np.linspace(-self.L / 2, self.L / 2, self.N, endpoint=False, dtype=np.float32)
        self.X, self.Y = np.meshgrid(x, x)
        self.dx  = float(x[1] - x[0])
        self.dx2 = self.dx ** 2

        # ── Momentum grid ─────────────────────────────────────────────────────
        kx = fftfreq(self.N, d=self.dx).astype(np.float32) * (2.0 * np.pi)
        Kx, Ky = np.meshgrid(kx, kx)
        self.K_sq        = (Kx ** 2 + Ky ** 2).astype(np.float32)
        self.noise_filter = np.exp(-0.04 * self.K_sq).astype(np.complex64)

        self.dt             = 0.001
        self.steps_per_frame = 20

        # ── Simulation state ──────────────────────────────────────────────────
        self.psi           = None
        self.V             = np.zeros((self.N, self.N), dtype=np.float32)
        self.is_running    = False
        self.bg            = None
        self.max_dens_cache = 1.0

        # ── Cached propagators ────────────────────────────────────────────────
        # All three are rebuilt together whenever _potential_dirty or gamma changes.
        self._exp_K      = None   # complex64, shape (N,N) – kinetic half-step
        self._exp_V_half = None   # complex64, shape (N,N) – potential half-step
        self._exp_V_full = None   # complex64, shape (N,N) – potential full step
        self._cached_gamma      = None   # gamma value used for the cached arrays
        self._potential_dirty   = True   # set True whenever V parameters change

        # ── Recording state ───────────────────────────────────────────────────
        self.is_recording    = False
        self.record_frames   = []
        self.record_count    = 0
        self.record_target   = 0
        self.record_filename = ""

        self._build_ui()

        # Invalidate potential cache whenever any potential control changes.
        # (g and gamma are handled live inside _evolve_steps.)
        for var in (self.typeV_var, self.v0_var, self.wV_var):
            var.trace_add('write', self._mark_potential_dirty)

        self.initialize_wavefunction()
        self._update_loop()

    # ──────────────────────────────────────────────────────────────────────────
    # UI helpers
    # ──────────────────────────────────────────────────────────────────────────

    def _mark_potential_dirty(self, *_):
        self._potential_dirty = True

    def _create_slider(self, parent, label_text, var, from_, to):
        lbl_frame = ttk.Frame(parent)
        lbl_frame.pack(fill=tk.X, pady=(1, 0))
        ttk.Label(lbl_frame, text=label_text).pack(side=tk.LEFT)

        entry_var = tk.StringVar(value=f"{var.get():.2f}")
        val_entry = ttk.Entry(lbl_frame, textvariable=entry_var, width=5,
                              font=("Arial", 9, "bold"), justify="right")
        val_entry.pack(side=tk.RIGHT)

        _lock = [False]

        def update_entry(*_):
            if not _lock[0]:
                _lock[0] = True
                entry_var.set(f"{var.get():.2f}")
                _lock[0] = False

        def update_slider(_=None):
            if not _lock[0]:
                _lock[0] = True
                try:
                    val = max(min(float(entry_var.get()), to), from_)
                    var.set(val)
                    entry_var.set(f"{val:.2f}")
                    parent.focus_set()
                except ValueError:
                    entry_var.set(f"{var.get():.2f}")
                _lock[0] = False

        var.trace_add('write', update_entry)
        val_entry.bind('<Return>',   update_slider)
        val_entry.bind('<FocusOut>', update_slider)

        def snap(val_str):
            val = float(val_str)
            if abs(val - round(val)) < 0.15:
                var.set(round(val))

        slider = ttk.Scale(parent, from_=from_, to=to, variable=var, command=snap)
        slider.pack(fill=tk.X, pady=(0, 2))
        return slider

    def _build_ui(self):
        # ── Left control panel ────────────────────────────────────────────────
        ctrl = ttk.Frame(self.root, width=320, padding=10)
        ctrl.pack(side=tk.LEFT, fill=tk.Y)
        ttk.Label(ctrl, text="2D Simulator Controls",
                  font=("Arial", 14, "bold")).pack(pady=5)

        self.tabs = ttk.Notebook(ctrl)
        self.tabs.pack(fill=tk.BOTH, expand=True, pady=5)
        tab_sys   = ttk.Frame(self.tabs, padding=10)
        tab_waves = ttk.Frame(self.tabs, padding=10)
        self.tabs.add(tab_sys,   text="System")
        self.tabs.add(tab_waves, text="Wavepackets")

        # System tab
        f_mode = ttk.LabelFrame(tab_sys, text="Evolution Mode", padding=5)
        f_mode.pack(fill=tk.X, pady=5)
        self.sim_mode_var = tk.StringVar(value="Pure GPE (Real Time Dynamics)")
        cb_mode = ttk.Combobox(f_mode, textvariable=self.sim_mode_var, state="readonly",
                               values=["Pure GPE (Real Time Dynamics)",
                                       "Thermal SGPE (Cooling & Temp)"])
        cb_mode.pack(fill=tk.X, pady=5)
        cb_mode.bind("<<ComboboxSelected>>", self._toggle_temp_slider)

        f_phys = ttk.LabelFrame(tab_sys, text="Physics", padding=5)
        f_phys.pack(fill=tk.X, pady=5)
        self.g_var = tk.DoubleVar(value=50.0)
        self._create_slider(f_phys, "Interaction (g):", self.g_var, -100.0, 200.0)

        self.frame_therm = ttk.LabelFrame(tab_sys, text="Thermal Control", padding=5)
        self.frame_therm.pack(fill=tk.X, pady=5)
        self.temp_var = tk.DoubleVar(value=50.0)
        self._create_slider(self.frame_therm, "Temperature (T):", self.temp_var, 0.0, 100.0)
        self._toggle_state(self.frame_therm, "disabled")

        f_pot = ttk.LabelFrame(tab_sys, text="External Potential 2D", padding=5)
        f_pot.pack(fill=tk.X, pady=5)
        self.typeV_var = tk.StringVar(value="None")
        ttk.Combobox(f_pot, textvariable=self.typeV_var, state="readonly",
                     values=["None", "Harmonic Bowl", "Optical Lattice",
                             "Central Pillar (Obstacle)", "Ring Trap (Annulus)",
                             "Double Well", "Circular Box Trap",
                             "Random Disorder"]).pack(fill=tk.X, pady=5)
        self.v0_var = tk.DoubleVar(value=50.0)
        self._create_slider(f_pot, "Strength (V0):",    self.v0_var, -200.0, 200.0)
        self.wV_var = tk.DoubleVar(value=4.0)
        self._create_slider(f_pot, "Scale/Radius (w):", self.wV_var,    0.5,  20.0)

        # Wavepackets tab
        f_w1 = ttk.LabelFrame(tab_waves, text="Wavepacket 1", padding=5)
        f_w1.pack(fill=tk.X, pady=(0, 5))
        self.x1_var  = tk.DoubleVar(value=-6.0)
        self.y1_var  = tk.DoubleVar(value=1.5)
        self.vx1_var = tk.DoubleVar(value=8.0)
        self.vy1_var = tk.DoubleVar(value=0.0)
        self.w1_var  = tk.DoubleVar(value=2.5)
        lim = self.L / 2
        self._create_slider(f_w1, "Pos X:",  self.x1_var,  -lim, lim)
        self._create_slider(f_w1, "Pos Y:",  self.y1_var,  -lim, lim)
        self._create_slider(f_w1, "Vel X:",  self.vx1_var, -20.0, 20.0)
        self._create_slider(f_w1, "Vel Y:",  self.vy1_var, -20.0, 20.0)
        self._create_slider(f_w1, "Width:",  self.w1_var,   0.5,  10.0)

        self.enable_w2_var = tk.BooleanVar(value=True)
        ttk.Checkbutton(tab_waves, text="Enable Wavepacket 2",
                        variable=self.enable_w2_var,
                        command=self._toggle_w2).pack(anchor=tk.W, pady=(5, 2))

        self.frame_w2 = ttk.LabelFrame(tab_waves, text="Wavepacket 2", padding=5)
        self.frame_w2.pack(fill=tk.X, pady=(0, 5))
        self.x2_var  = tk.DoubleVar(value=6.0)
        self.y2_var  = tk.DoubleVar(value=-1.5)
        self.vx2_var = tk.DoubleVar(value=-8.0)
        self.vy2_var = tk.DoubleVar(value=0.0)
        self.w2_var  = tk.DoubleVar(value=2.5)
        self._create_slider(self.frame_w2, "Pos X:",  self.x2_var,  -lim, lim)
        self._create_slider(self.frame_w2, "Pos Y:",  self.y2_var,  -lim, lim)
        self._create_slider(self.frame_w2, "Vel X:",  self.vx2_var, -20.0, 20.0)
        self._create_slider(self.frame_w2, "Vel Y:",  self.vy2_var, -20.0, 20.0)
        self._create_slider(self.frame_w2, "Width:",  self.w2_var,   0.5,  10.0)

        # Buttons
        b = ttk.Frame(ctrl)
        b.pack(fill=tk.X, pady=(10, 10))
        ttk.Button(b, text="Initialize", command=self.initialize_wavefunction).grid(row=0, column=0, padx=2, pady=2, sticky="ew")
        ttk.Button(b, text="Start",      command=self.start_sim).grid(row=0, column=1, padx=2, pady=2, sticky="ew")
        ttk.Button(b, text="Stop",       command=self.stop_sim ).grid(row=0, column=2, padx=2, pady=2, sticky="ew")

        gif_row = ttk.Frame(b)
        gif_row.grid(row=1, column=0, columnspan=3, pady=(10, 2), sticky="ew")
        ttk.Label(gif_row, text="Frames:").pack(side=tk.LEFT, padx=(0, 5))
        self.gif_frames_var = tk.IntVar(value=200)
        ttk.Entry(gif_row, textvariable=self.gif_frames_var, width=5).pack(side=tk.LEFT)
        self.btn_gif = ttk.Button(gif_row, text="Save GIF (Live Record)",
                                  command=self.start_live_recording)
        self.btn_gif.pack(side=tk.LEFT, padx=(10, 0), expand=True, fill=tk.X)
        b.columnconfigure((0, 1, 2), weight=1)

        # ── Right: dual plot ──────────────────────────────────────────────────
        plot_frame = ttk.Frame(self.root)
        plot_frame.pack(side=tk.RIGHT, fill=tk.BOTH, expand=True)

        self.fig = Figure(figsize=(12, 6), dpi=100)
        self.fig.patch.set_facecolor('#1e1e1e')

        ext = [-self.L / 2, self.L / 2, -self.L / 2, self.L / 2]

        self.ax_dens = self.fig.add_subplot(121)
        self.ax_dens.set_facecolor('black')
        self.ax_dens.set_title(r'Density $|\psi|^2$', color='white', fontsize=16)
        self.ax_dens.tick_params(colors='white')
        self.img_dens = self.ax_dens.imshow(
            np.zeros((self.N, self.N)), extent=ext, origin='lower',
            cmap='magma', animated=True, vmin=0.0, vmax=1.0)
        cbar1 = self.fig.colorbar(self.img_dens, ax=self.ax_dens,
                                  fraction=0.046, pad=0.04)
        cbar1.ax.yaxis.set_tick_params(color='white', labelcolor='white')

        self.ax_phase = self.fig.add_subplot(122)
        self.ax_phase.set_facecolor('black')
        self.ax_phase.set_title(r'Quantum Phase $\theta$', color='white', fontsize=16)
        self.ax_phase.tick_params(colors='white')

        # FIX: replaces deprecated cm.get_cmap()
        cmap_phase = mpl_cm['twilight_shifted'].copy()
        cmap_phase.set_bad('black')
        self.img_phase = self.ax_phase.imshow(
            np.zeros((self.N, self.N)), extent=ext, origin='lower',
            cmap=cmap_phase, animated=True, vmin=-np.pi, vmax=np.pi)
        cbar2 = self.fig.colorbar(self.img_phase, ax=self.ax_phase,
                                  fraction=0.046, pad=0.04,
                                  ticks=[-np.pi, 0, np.pi])
        cbar2.ax.set_yticklabels([r'$-\pi$', '0', r'$\pi$'])
        cbar2.ax.yaxis.set_tick_params(color='white', labelcolor='white')

        self.fig.tight_layout()
        self.canvas = FigureCanvasTkAgg(self.fig, master=plot_frame)
        self.canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)

    # ──────────────────────────────────────────────────────────────────────────
    # State toggles
    # ──────────────────────────────────────────────────────────────────────────

    def _toggle_state(self, widget, state):
        try:
            widget.configure(state=state)
        except tk.TclError:
            pass
        for child in widget.winfo_children():
            self._toggle_state(child, state)

    def _toggle_w2(self):
        self._toggle_state(self.frame_w2,
                           "normal" if self.enable_w2_var.get() else "disabled")

    def _toggle_temp_slider(self, _=None):
        state = "normal" if "Thermal SGPE" in self.sim_mode_var.get() else "disabled"
        self._toggle_state(self.frame_therm, state)

    # ──────────────────────────────────────────────────────────────────────────
    # Physics helpers
    # ──────────────────────────────────────────────────────────────────────────

    def update_potential_array(self):
        v_type = self.typeV_var.get()
        v0     = self.v0_var.get()
        w      = self.wV_var.get()
        r      = np.hypot(self.X, self.Y)

        if v_type == "Harmonic Bowl":
            V = v0 * ((self.X / 10.0) ** 2 + (self.Y / 10.0) ** 2)
        elif v_type == "Optical Lattice":
            V = v0 * (np.sin(np.pi * self.X / w) ** 2
                    + np.sin(np.pi * self.Y / w) ** 2)
        elif v_type == "Central Pillar (Obstacle)":
            V = np.where(r <= w, v0, 0.0)
        elif v_type == "Ring Trap (Annulus)":
            V = v0 * ((r - w) / 2.0) ** 2
        elif v_type == "Double Well":
            V = v0 * (((self.X ** 2 - w ** 2) ** 2) / (w ** 4 + 1e-6)
                    + (self.Y / (w + 1e-6)) ** 2)
        elif v_type == "Circular Box Trap":
            V = np.where(r > w, v0, 0.0)
        elif v_type == "Random Disorder":
            rng = np.random.default_rng(42)
            V = np.zeros_like(self.X)
            for _ in range(6):
                kx    = rng.uniform(-2, 2)
                ky    = rng.uniform(-2, 2)
                phase = rng.uniform(0, 2 * np.pi)
                V    += np.sin(kx * self.X + ky * self.Y + phase)
            lo, hi = V.min(), V.max()
            V = v0 * (V - lo) / (hi - lo)
        else:
            V = np.zeros((self.N, self.N), dtype=np.float32)

        self.V = V.astype(np.float32)

    def _get_gamma(self):
        return 0.05 if "Thermal SGPE" in self.sim_mode_var.get() else 0.0

    def _rebuild_propagators(self):
        """Recompute exp_K, exp_V_half, exp_V_full from current gamma and V."""
        gamma  = self._get_gamma()
        c_half = np.complex64(-(1j + gamma) * 0.5 * self.dt)
        c_full = np.complex64(-(1j + gamma) * self.dt)

        self._exp_K      = np.exp(c_half * self.K_sq).astype(np.complex64)
        self._exp_V_half = np.exp(c_half * self.V).astype(np.complex64)
        self._exp_V_full = np.exp(c_full * self.V).astype(np.complex64)
        self._cached_gamma    = gamma
        self._potential_dirty = False

    def create_2d_wavepacket(self, x0, y0, vx, vy, w):
        r_sq = (self.X - x0) ** 2 + (self.Y - y0) ** 2
        return (np.exp(-0.5 * r_sq / w ** 2)
                * np.exp(1j * (vx * self.X + vy * self.Y))).astype(np.complex64)

    # ──────────────────────────────────────────────────────────────────────────
    # Simulation control
    # ──────────────────────────────────────────────────────────────────────────

    def initialize_wavefunction(self):
        self.stop_sim()
        self.update_potential_array()
        self._potential_dirty = True          # force propagator rebuild next step

        psi = self.create_2d_wavepacket(
            self.x1_var.get(), self.y1_var.get(),
            self.vx1_var.get(), self.vy1_var.get(), self.w1_var.get())
        if self.enable_w2_var.get():
            psi += self.create_2d_wavepacket(
                self.x2_var.get(), self.y2_var.get(),
                self.vx2_var.get(), self.vy2_var.get(), self.w2_var.get())

        norm = np.sqrt((psi.real ** 2 + psi.imag ** 2).sum() * self.dx2)
        if norm > 0:
            psi /= norm
        self.psi = psi

        self.max_dens_cache = float((psi.real ** 2 + psi.imag ** 2).max())
        self.img_dens.set_clim(vmin=0, vmax=self.max_dens_cache * 1.5)

        self.bg = None
        self._full_redraw()

    def start_sim(self):
        if self.psi is None or self.bg is None:
            self.initialize_wavefunction()
        if not self.is_running:
            self.is_running = True

    def stop_sim(self):
        self.is_running = False

    # ──────────────────────────────────────────────────────────────────────────
    # Numerical evolution  (Strang split-step, no duplicated FFT block)
    # ──────────────────────────────────────────────────────────────────────────

    def _evolve_steps(self):
        gamma = self._get_gamma()

        # Rebuild propagators only when something actually changed
        if self._potential_dirty or gamma != self._cached_gamma:
            if self._potential_dirty:
                self.update_potential_array()
            self._rebuild_propagators()

        exp_K      = self._exp_K
        exp_V_half = self._exp_V_half
        exp_V_full = self._exp_V_full

        g       = np.complex64(self.g_var.get())
        dt      = self.dt
        N       = self.N
        dx2     = self.dx2
        c_half  = np.complex64(-(1j + gamma) * 0.5 * dt)
        c_full  = np.complex64(-(1j + gamma) * dt)
        g_ch    = c_half * g
        g_cf    = c_full * g

        T        = float(self.temp_var.get()) if gamma > 0 else 0.0
        do_noise = gamma > 0 and T > 0
        do_norm  = gamma > 0
        noise_amp = np.float32((0.00015 * T) * N) if do_noise else 0.0

        last = self.steps_per_frame - 1
        psi  = self.psi

        # ── Initial half real-space kick ──────────────────────────────────────
        dens  = psi.real ** 2 + psi.imag ** 2
        psi  *= exp_V_half * np.exp(g_ch * dens)

        # ── Unified loop: removes the duplicated post-loop FFT block ──────────
        for step in range(self.steps_per_frame):
            # Kinetic (k-space) half-step
            psi_k  = fft2(psi, workers=-1)
            if do_noise:
                psi_k += noise_amp * (
                    np.random.randn(N, N).astype(np.float32)
                    + 1j * np.random.randn(N, N).astype(np.float32)
                ) * self.noise_filter
            psi_k *= exp_K
            psi    = ifft2(psi_k, workers=-1)

            if do_norm:
                norm = np.sqrt((psi.real ** 2 + psi.imag ** 2).sum() * dx2)
                if norm > 0:
                    psi /= norm

            # Real-space kick: half on the final step, full otherwise
            dens = psi.real ** 2 + psi.imag ** 2
            if step < last:
                psi *= exp_V_full * np.exp(g_cf * dens)
            else:
                psi *= exp_V_half * np.exp(g_ch * dens)

        self.psi = psi

    # ──────────────────────────────────────────────────────────────────────────
    # Rendering
    # ──────────────────────────────────────────────────────────────────────────

    def _update_image_data(self):
        dens  = self.psi.real ** 2 + self.psi.imag ** 2
        phase = np.where(dens > 0.01 * self.max_dens_cache,
                         np.angle(self.psi), np.nan)
        self.img_dens.set_data(dens)
        self.img_phase.set_data(phase)

    def _full_redraw(self):
        """Synchronous full render + background capture for blitting."""
        if self.psi is None:
            return
        self._update_image_data()
        self.canvas.draw()                                      # single draw – no draw_idle
        self.bg = self.canvas.copy_from_bbox(self.fig.bbox)    # background captured right after

    def _fast_redraw(self):
        """Blit only the two image artists – much faster than a full draw."""
        if self.bg is None:
            self._full_redraw()
            return
        self._update_image_data()
        try:
            self.canvas.restore_region(self.bg)
            self.ax_dens.draw_artist(self.img_dens)
            self.ax_phase.draw_artist(self.img_phase)
            self.canvas.blit(self.fig.bbox)
            self.canvas.flush_events()
        except Exception:
            self._full_redraw()

    # ──────────────────────────────────────────────────────────────────────────
    # Live GIF recording
    # ──────────────────────────────────────────────────────────────────────────

    def start_live_recording(self):
        if self.is_recording:
            messagebox.showwarning("Recording", "Already recording!")
            return
        try:
            frames_to_save = self.gif_frames_var.get()
            if frames_to_save < 1:
                raise ValueError
        except ValueError:
            messagebox.showerror("Error", "Enter a valid positive integer for Frames.")
            return

        filename = filedialog.asksaveasfilename(
            defaultextension=".gif",
            initialfile="2d_sgpe_simulation.gif",
            filetypes=[("GIF Image", "*.gif"), ("All Files", "*.*")],
            title="Save Animation As…")
        if not filename:
            return

        self.record_filename = filename
        self.record_target   = frames_to_save
        self.record_count    = 0
        self.record_frames   = []
        self.is_recording    = True

        if not self.is_running:
            self.start_sim()

    def _capture_frame(self):
        """
        FIX: single canvas.draw() then buffer_rgba() – replaces the deprecated
        tostring_rgb() and eliminates the redundant second draw call.
        """
        self._update_image_data()
        self.canvas.draw()
        # buffer_rgba() returns a memoryview of shape (H, W, 4) in RGBA order
        buf   = np.asarray(self.canvas.renderer.buffer_rgba())
        return buf[:, :, :3].copy()    # drop alpha channel → RGB uint8

    def _finish_recording(self):
        self.is_recording = False
        self.btn_gif.config(text="⚙️ Encoding GIF…")

        frames_to_encode = self.record_frames
        filename         = self.record_filename
        self.record_frames = []

        def encode():
            try:
                imgs = [Image.fromarray(f) for f in frames_to_encode]
                imgs[0].save(filename, save_all=True, append_images=imgs[1:],
                             duration=33, loop=0)
                self.root.after(0, lambda: self.btn_gif.config(
                    text="Save GIF (Live Record)"))
                self.root.after(0, lambda: messagebox.showinfo(
                    "Success", f"Saved to:\n{filename}"))
            except Exception as e:
                self.root.after(0, lambda: self.btn_gif.config(
                    text="Save GIF (Live Record)"))
                self.root.after(0, lambda e=e: messagebox.showerror(
                    "Error", f"GIF encode failed:\n{e}"))

        threading.Thread(target=encode, daemon=True).start()

    # ──────────────────────────────────────────────────────────────────────────
    # Main update loop
    # ──────────────────────────────────────────────────────────────────────────

    def _update_loop(self):
        if self.is_running:
            self._evolve_steps()

            if self.is_recording:
                frame = self._capture_frame()
                self.record_frames.append(frame)
                self.record_count += 1
                self.btn_gif.config(
                    text=f"🔴 Rec: {self.record_count} / {self.record_target}")
                if self.record_count >= self.record_target:
                    self._finish_recording()
            else:
                if self.bg is None:
                    self._full_redraw()
                self._fast_redraw()

        self.root.after(33, self._update_loop)


if __name__ == "__main__":
    root = tk.Tk()
    app  = GPE2DSimulatorApp(root)
    root.mainloop()